# M2 — Click Propensity Model

Estimates the probability that a user clicks on a product, in order to rank each user's products; that ranking is used in notebook 08.

Two formulations of this probability are compared:

1. Flat model: a single model estimates the click probability of the (user, product) pair.
2. Hierarchical model: in two steps, sector probability × product-within-sector probability.

After the comparison (later in the notebook), the flat model outperforms the hierarchical one across all algorithms, so it is the one adopted. The hierarchical model is kept as an interpretable variant that decomposes the decision into sector and product.

A large share of the user variables are synthetic (derived from area averages in notebook 03) and carry little signal. The relevant signal comes from the sector, the product and the subject-line text.


## 0 · Libraries and paths


In [ ]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import lightgbm as lgb
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import roc_auc_score, average_precision_score
from scipy.stats import wilcoxon

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")

# Adds src/ to the path and reuses the shared helpers (same pattern as 01-04)
_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(_root / "src"))
from tfm.utils import find_project_root  # noqa: E402
from tfm.calibration import correct_prior  # noqa: E402

ROOT_PATH = find_project_root()
PROCESSED_PATH = ROOT_PATH / "data" / "processed"
EXTERNAL_PATH = ROOT_PATH / "data" / "external_clean"

np.random.seed(42)   # seed so the results are reproducible
print("Raíz del proyecto:", ROOT_PATH)

## 1 · Loading the data


In [ ]:
users = pd.read_csv(PROCESSED_PATH / "users.csv", dtype={"cp_num": str})
events = pd.read_csv(PROCESSED_PATH / "events.csv")
products = pd.read_csv(PROCESSED_PATH / "products.csv")
demo_segments = pd.read_csv(PROCESSED_PATH / "users_demo_segments.csv")
tabla_marcas = pd.read_csv(EXTERNAL_PATH / "tabla_marcas.csv", sep=None, engine="python")

# Keep only click and open events, and build the target:
#   target = 1 if there was a click, 0 if the email was only opened
events = events[events["event_type"].isin(["click", "open"])].copy()
events["target"] = (events["event_type"] == "click").astype(int)

print("users   :", users.shape)
print("events  :", events.shape)
print("products:", products.shape)

## 2 · Click rate (prior)

`rho_train` is the proportion of clicks in the sample (inflated by the 1:1 sampling). `rho_real` ≈ 0.02 is the proportion of clicks in production; it is used at the end to correct the probabilities towards their realistic level.


In [ ]:
rho_train = float(events["target"].mean())
rho_real = 0.02

print("Nº de eventos :", len(events))
print("Nº de clicks  :", int(events["target"].sum()))
print("rho_train (tasa de clicks en la muestra):", round(rho_train, 4))
print("rho_real  (tasa de clicks real)         :", rho_real)

## 3 · Preparing the USER variables

The categorical columns are encoded as numbers. Age is binned into `age_cat` using the same bands as the production system: 0 = 18-24, 1 = 25-34, 2 = 35-44, 3 = 45-54, 4 = 55-64, 5 = 65+.


In [ ]:
u = users.copy()

# Gender: Male = 1, Female = 0
u["gender_enc"] = (u["gender"] == "H").astype(int)

# Employment status: mapped to a 0, 1, 2 scale
mapa_laboral = {"employed": 2, "unemployed": 1, "inactive": 0}
u["labor_status_enc"] = u["labor_status"].map(mapa_laboral).fillna(0).astype(int)

# Marital status: mapped to numbers
mapa_civil = {"soltero": 0, "divorciado": 1, "viudo": 2, "casado": 3}
u["civil_status_enc"] = u["civil_status"].map(mapa_civil).fillna(0).astype(int)

# Car ownership: True/False -> 1/0
u["tiene_coche_enc"] = u["tiene_coche"].astype(int)

# Number of rooms: small/medium/large -> 1/2/3
mapa_habitaciones = {"menos_3_hab": 1, "3_a_6_hab": 2, "7_mas_hab": 3}
u["num_room_enc"] = u["num_room"].map(mapa_habitaciones).fillna(2).astype(int)


# Household size: take the first digit of the text ("3 personas" -> 3)
def tamano_hogar(texto):
    if pd.isna(texto):
        return 3
    texto = str(texto).strip()
    if texto[:1].isdigit() and texto[0] != "0":
        return int(texto[0])
    return 5


u["size_hogar_enc"] = u["size_hogar"].apply(tamano_hogar).clip(1, 5)


# Age -> age group (age_cat)
def edad_a_grupo(edad):
    if pd.isna(edad):
        return -1          # unknown age
    if edad < 25:
        return 0           # 18-24
    elif edad < 35:
        return 1           # 25-34
    elif edad < 45:
        return 2           # 35-44
    elif edad < 55:
        return 3           # 45-54
    elif edad < 65:
        return 4           # 55-64
    else:
        return 5           # 65+


u["age_cat"] = u["age"].apply(edad_a_grupo).astype(int)

# Add the demographic cluster (comes from notebook 05)
u = u.merge(demo_segments[["id_user", "demo_cluster"]], on="id_user", how="left")
u["demo_cluster"] = u["demo_cluster"].fillna(-1).astype(int)

# List of user variables the model will use
USER_FEATS = [
    "age_cat", "gender_enc", "labor_status_enc", "civil_status_enc",
    "tiene_coche_enc", "size_hogar_enc", "num_room_enc",
    "ipa_class", "mun_type", "distance_type", "demo_cluster",
]
print("Variables de usuario:", USER_FEATS)
print("\nReparto por grupo de edad:")
print(u["age_cat"].value_counts().sort_index())

## 4 · Preparing the PRODUCT variables

A click occurs between a user and a specific email: it is a property of the (user, product) pair. A model using only user variables would give the same value for all of that user's products and could not rank them. To distinguish products there are two options: (A) one model per category or (B) a single model with the product as an input variable. Both are compared (Variant A = user only; Variant B = user + product).

Product variables used:
- Category (`product_new`) and sector.
- `cpl` (cost per lead).
- Marketing attributes from `tabla_marcas.csv` (urgency, price sensitivity, etc.). The join by brand covers only 27 % of events, so they are aggregated at sector level (100 % coverage).
- Subject-line embeddings (section 5).


In [ ]:
# 4.1 · Marketing attributes -> numbers, and mean per sector

# Columns of tabla_marcas we care about (those that exist)
posibles_attrs = ["Urgencia", "Racionalidad", "RiesgoPercibido", "CicloDecision",
                  "Implicación", "Necesidad", "SensibilidadPrecio", "CompetenciaAlta", "PrecioMedio"]
attr_cols = [c for c in posibles_attrs if c in tabla_marcas.columns]

# Dictionary to convert text ("Alta", "Baja"...) to a number
texto_a_numero = {
    "muy baja": 0, "baja": 1, "baja-media": 1, "media": 2, "medio": 2,
    "media-alta": 3, "alta": 4, "muy alta": 5, "alto": 4, "bajo": 1,
    "no": 0, "si": 1, "sí": 1, "corto": 0, "medio-largo": 2, "largo": 3,
}

tm = tabla_marcas.copy()
attr_num_cols = []   # names of the numeric columns we are creating
for col in attr_cols:
    nueva = col + "_num"
    if tm[col].dtype == object:
        # text -> lowercase -> number
        tm[nueva] = tm[col].astype(str).str.strip().str.lower().map(texto_a_numero)
    else:
        tm[nueva] = pd.to_numeric(tm[col], errors="coerce")
    attr_num_cols.append(nueva)

# Normalise the sector name so the tables can be joined
nombre_col_sector = [c for c in tabla_marcas.columns if c.lower() == "sector"][0]
tm["sector_norm"] = tm[nombre_col_sector].astype(str).str.strip().str.lower()

# Mean of each attribute per sector
sector_attrs = tm.groupby("sector_norm")[attr_num_cols].mean().reset_index()
print("Atributos por sector listos:", sector_attrs.shape)

In [ ]:
# 4.2 · Preparing the products table

p = products.copy()
p["sector_norm"] = p["sector"].astype(str).str.strip().str.lower()
p["cpl"] = pd.to_numeric(p["cpl"], errors="coerce")

# Convert category and sector into numeric codes
p["prod_cat_code"] = p["product_new"].astype("category").cat.codes
p["sector_code"] = p["sector"].astype("category").cat.codes

# Join the marketing attributes by sector
p = p.merge(sector_attrs, on="sector_norm", how="left")

PROD_NUM_FEATS = ["cpl"] + attr_num_cols
print("Variables numéricas de producto:", PROD_NUM_FEATS)

## 5 · Subject-line embeddings

The email subject is turned into a vector with `sentence-transformers` (384 dimensions) and the raw per-product vectors are saved (`subject_emb_raw.csv`) for reuse.

The PCA reduction to 16 components is performed inside each cross-validation fold (section 8), fitting the PCA on the training products only. Fitting the PCA once over all products would introduce leakage, since the components would have seen validation data.


In [ ]:
ruta_emb = PROCESSED_PATH / "subject_emb_raw.csv"
N_EMB = 16   # final PCA components; the PCA is fitted WITHIN each fold (section 8), no leakage

if ruta_emb.exists():
    # Already computed: load the RAW embeddings (384 dimensions)
    emb_df = pd.read_csv(ruta_emb)
    print("Embeddings crudos cargados:", emb_df.shape)
else:
    # Does not exist: compute them once (language model) and save them raw
    from sentence_transformers import SentenceTransformer

    modelo_texto = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
    textos = p["subject_clean"].fillna("").tolist()
    vectores = np.asarray(modelo_texto.encode(textos, show_progress_bar=False), dtype=np.float32)

    emb_df = pd.DataFrame(vectores, columns=[f"emb_{i}" for i in range(vectores.shape[1])])
    emb_df["id_product"] = p["id_product"].values
    emb_df.to_csv(ruta_emb, index=False)
    print("Embeddings crudos calculados y guardados:", emb_df.shape)

# Product -> raw embedding table (the PCA is applied later per fold, no leakage)
EMB_RAW_COLS = [c for c in emb_df.columns if c.startswith("emb_")]
emb_raw = emb_df.drop_duplicates("id_product").set_index("id_product")[EMB_RAW_COLS]
print(f"Embedding crudo por producto: {emb_raw.shape}  ->  PCA a {N_EMB} comp. dentro de cada fold")

## 6 · Bringing everything together into an events table

Each row is an event (an email sent to a user): user variables, product variables and the click label. The numerical gaps are not imputed: LightGBM handles `NaN` values natively (it learns the direction per fold) and the subject-line embeddings are added per fold (section 8).


In [ ]:
# Join events + user + product (no embeddings: they are added per fold in section 8)
df = events[["id_event", "id_user", "id_product", "target"]].copy()
df = df.merge(u[["id_user"] + USER_FEATS], on="id_user", how="left")

cols_producto = ["id_product", "sector", "product_new", "sector_code", "prod_cat_code"] + PROD_NUM_FEATS
df = df.merge(p[cols_producto], on="id_product", how="left")

# Drop events with no product. We do NOT impute the numerical gaps: LightGBM handles NaN
# natively (no leakage). Previously the GLOBAL median was imputed, which mixed train and validation.
df = df.dropna(subset=["sector"]).reset_index(drop=True)

print("Tabla final:", df.shape)
print("Tasa de clicks:", round(df["target"].mean(), 4))

## 7 · Setting up the validation (GroupKFold by user)

The data is split into 5 folds grouped by user: the same user does not appear in both training and validation at once. Each row is predicted by a model that has not seen it (out-of-fold, OOF). Two bootstrap functions are also prepared for the confidence intervals.


In [ ]:
from sklearn.decomposition import PCA

N_FOLDS = 5
N_BOOT = 1000   # number of bootstrap repetitions

# LightGBM parameters. num_leaves=15 = simple model (with weak signal it generalises better).
LGBM_PARAMS = {
    "objective": "binary", "n_estimators": 300, "learning_rate": 0.05,
    "num_leaves": 15, "min_child_samples": 20, "subsample": 0.8,
    "colsample_bytree": 0.8, "random_state": 42, "verbose": -1, "n_jobs": -1,
}

y = df["target"].values            # what we want to predict (0/1)
groups = df["id_user"].values      # to group by user

# Which validation fold each row belongs to (used in the per-fold test)
sgkf = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
fold_de_cada_fila = np.full(len(y), -1)
for numero_fold, (idx_train, idx_val) in enumerate(sgkf.split(np.zeros(len(y)), y, groups)):
    fold_de_cada_fila[idx_val] = numero_fold

# Categorical columns (LightGBM treats them specially)
CAT_COLS = ["demo_cluster", "sector_code", "prod_cat_code"]

# Raw embeddings aligned by row (for the per-fold PCA, no leakage)
_prod_pos = {pid: i for i, pid in enumerate(emb_raw.index)}
EMB_RAW_MAT = emb_raw.values.astype(np.float32)            # (n_products, 384)
ROW_PROD = df["id_product"].map(_prod_pos).values          # row -> product index in EMB_RAW_MAT


def predecir_oof(feature_cols, use_emb=False, params=LGBM_PARAMS):
    """Trains 5 models (one per fold) and returns each row's prediction
    made by the model that did NOT see it (out-of-fold).

    If use_emb=True, the subject-line embeddings are reduced with a PCA FITTED ONLY on the
    training products of each fold (no leakage) and added as columns to the model."""
    X = df[feature_cols].values.astype(float)
    cat_idx = [feature_cols.index(c) for c in CAT_COLS if c in feature_cols]
    oof = np.zeros(len(y))
    cv = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
    for idx_train, idx_val in cv.split(X, y, groups):
        Xtr, Xva = X[idx_train], X[idx_val]
        if use_emb:
            # PCA fitted ONLY on the products present in this fold's training set
            prods_tr = np.unique(ROW_PROD[idx_train])
            k = min(N_EMB, len(prods_tr) - 1)
            pca = PCA(n_components=k, random_state=42).fit(EMB_RAW_MAT[prods_tr])
            emb_row = pca.transform(EMB_RAW_MAT)[ROW_PROD]   # (n_rows, k)
            Xtr = np.hstack([Xtr, emb_row[idx_train]])
            Xva = np.hstack([Xva, emb_row[idx_val]])
        modelo = lgb.LGBMClassifier(**params)
        modelo.fit(Xtr, y[idx_train], categorical_feature=cat_idx)
        oof[idx_val] = modelo.predict_proba(Xva)[:, 1]
    return oof


def pr_auc(prob):
    """PR-AUC of a set of probabilities against the real target."""
    return average_precision_score(y, prob)


def bootstrap_pr_auc(prob, n=N_BOOT):
    """Mean and 95% confidence interval of the PR-AUC, resampling at random."""
    rng = np.random.RandomState(42)
    idx_pos = np.where(y == 1)[0]
    idx_neg = np.where(y == 0)[0]
    valores = []
    for _ in range(n):
        muestra = np.concatenate([
            rng.choice(idx_pos, len(idx_pos), replace=True),
            rng.choice(idx_neg, len(idx_neg), replace=True),
        ])
        valores.append(average_precision_score(y[muestra], prob[muestra]))
    valores = np.array(valores)
    return valores.mean(), np.percentile(valores, 2.5), np.percentile(valores, 97.5)


def bootstrap_diferencia(prob_a, prob_b, n=N_BOOT):
    """PR-AUC difference between two models (A - B) with its 95% CI and p-value."""
    rng = np.random.RandomState(42)
    idx_pos = np.where(y == 1)[0]
    idx_neg = np.where(y == 0)[0]
    diferencias = []
    for _ in range(n):
        muestra = np.concatenate([
            rng.choice(idx_pos, len(idx_pos), replace=True),
            rng.choice(idx_neg, len(idx_neg), replace=True),
        ])
        ap_a = average_precision_score(y[muestra], prob_a[muestra])
        ap_b = average_precision_score(y[muestra], prob_b[muestra])
        diferencias.append(ap_a - ap_b)
    diferencias = np.array(diferencias)
    # p-value: what fraction of the differences fall on the other side of 0
    p_valor = 2 * min((diferencias <= 0).mean(), (diferencias >= 0).mean())
    return diferencias.mean(), np.percentile(diferencias, 2.5), np.percentile(diferencias, 97.5), min(p_valor, 1.0)


print(f"{N_FOLDS} folds listos. num_leaves = {LGBM_PARAMS['num_leaves']}  | PCA de embeddings por fold (N_EMB={N_EMB})")

In [ ]:
# --- Data leakage: validation BY EVENT (with leakage) vs BY USER ---
# Model with ONLY the user variables (demographics). As these are identical across all events
# of the same user, a CV by event puts the same user in train and validation -> it memorises.
from sklearn.model_selection import StratifiedKFold

X_demo = df[USER_FEATS].values.astype(float)
cat_demo = [USER_FEATS.index("demo_cluster")]

def _oof_demo(splits):
    oof = np.zeros(len(y))
    for idx_tr, idx_va in splits:
        m = lgb.LGBMClassifier(**LGBM_PARAMS)
        m.fit(X_demo[idx_tr], y[idx_tr], categorical_feature=cat_demo)
        oof[idx_va] = m.predict_proba(X_demo[idx_va])[:, 1]
    return oof

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
auc_evento = roc_auc_score(y, _oof_demo(list(skf.split(X_demo, y))))
sgkf2 = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
auc_usuario = roc_auc_score(y, _oof_demo(list(sgkf2.split(X_demo, y, groups))))

print("Modelo demografico (solo variables de usuario):")
print(f"  AUC-ROC validacion POR EVENTO  (con fuga):    {auc_evento:.3f}")
print(f"  AUC-ROC validacion POR USUARIO (GroupKFold):  {auc_usuario:.3f}")
print(f"  -> la fuga infla el AUC en {auc_evento - auc_usuario:+.3f}")


## 8 · Level 1 — click probability by SECTOR


In [ ]:
# Features = user variables + the sector
feats_nivel1 = USER_FEATS + ["sector_code"]
p_sector = predecir_oof(feats_nivel1)

print("Nivel 1 (sector):")
print("  PR-AUC :", round(pr_auc(p_sector), 4))
print("  ROC-AUC:", round(roc_auc_score(y, p_sector), 4))

## 9 · Level 2 — Variant A (production replica)

As in the production system: **one model per category**, using **user variables only**.
Categories with very little data use their mean click rate (their "prior").


In [ ]:
feats_usuario = USER_FEATS
cat_idx_usuario = [feats_usuario.index("demo_cluster")]
X_usuario = df[feats_usuario].values.astype(float)

p_cond_A = np.full(len(df), np.nan)   # here we store the Level 2-A prediction

# Iterate over each product category
for categoria, indices in df.groupby("product_new").groups.items():
    indices = np.array(list(indices))
    y_cat = y[indices]
    n_clicks = y_cat.sum()
    n_noclicks = (y_cat == 0).sum()
    n_usuarios = len(np.unique(groups[indices]))

    # Check whether there is enough data to train in this category
    if n_clicks < 25 or n_noclicks < 25 or n_usuarios < N_FOLDS:
        # No: use the category's mean click rate
        p_cond_A[indices] = y_cat.mean()
    else:
        # Yes: train a model just for this category (OOF, grouping by user)
        cv = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
        for idx_tr, idx_va in cv.split(X_usuario[indices], y_cat, groups[indices]):
            modelo = lgb.LGBMClassifier(**LGBM_PARAMS)
            modelo.fit(X_usuario[indices][idx_tr], y_cat[idx_tr], categorical_feature=cat_idx_usuario)
            p_cond_A[indices[idx_va]] = modelo.predict_proba(X_usuario[indices][idx_va])[:, 1]

print("Nivel 2 A (un modelo por categoría, solo usuario):")
print("  PR-AUC:", round(pr_auc(p_cond_A), 4))

## 10 · Level 2 — Variant B (improvement: adds PRODUCT variables)

Here we train **a single model** that sees user **and** product variables (category, cpl,
marketing attributes and subject-line embeddings). It thus learns the user×product combination.


In [ ]:
feats_nivel2B = USER_FEATS + ["sector_code", "prod_cat_code"] + PROD_NUM_FEATS
p_cond_B = predecir_oof(feats_nivel2B, use_emb=True)   # embeddings with per-fold PCA (no leakage)

print("Nivel 2 B (usuario + producto + embeddings del asunto, PCA por fold):")
print("  PR-AUC :", round(pr_auc(p_cond_B), 4))
print("  ROC-AUC:", round(roc_auc_score(y, p_cond_B), 4))

## 11 · Final score and A vs B comparison

Final score = `p_sector × p_producto`. We compare A, B and a **popularity baseline** (recommending the
most-clicked item of each category). We look at PR-AUC with its interval, the B−A difference and a Wilcoxon test.


In [ ]:
score_A = p_sector * p_cond_A
score_B = p_sector * p_cond_B

# Popularity baseline: the mean click rate of each category
popularidad = df.groupby("product_new")["target"].transform("mean").values

# Comparison table with confidence intervals
modelos = [
    ("Baseline popularidad", popularidad),
    ("Nivel 1 (sector)", p_sector),
    ("A · producción", score_A),
    ("B · mejora", score_B),
]
filas = []
for nombre, prob in modelos:
    media, lo, hi = bootstrap_pr_auc(prob)
    filas.append({
        "modelo": nombre, "PR_AUC": pr_auc(prob),
        "IC95_lo": lo, "IC95_hi": hi, "ROC_AUC": roc_auc_score(y, prob),
    })
tabla = pd.DataFrame(filas)
print("=== PR-AUC con intervalo de confianza 95% ===")
print(tabla.round(4).to_string(index=False))

# B - A difference and significance test
dif, dif_lo, dif_hi, p_val = bootstrap_diferencia(score_B, score_A)
print(f"\nDiferencia PR-AUC (B - A): {dif:+.4f}   IC95% [{dif_lo:+.4f}, {dif_hi:+.4f}]   p≈{p_val:.4f}")
if dif_lo > 0:
    print("  -> B mejora a A de forma significativa.")
else:
    print("  -> sin diferencia significativa.")

# Per-fold Wilcoxon test (compares B and A in each of the 5 folds)
ap_A_por_fold = []
ap_B_por_fold = []
for f in range(N_FOLDS):
    filas_fold = (fold_de_cada_fila == f)
    ap_A_por_fold.append(average_precision_score(y[filas_fold], score_A[filas_fold]))
    ap_B_por_fold.append(average_precision_score(y[filas_fold], score_B[filas_fold]))
try:
    _, p_wilcoxon = wilcoxon(ap_B_por_fold, ap_A_por_fold)
    print(f"Wilcoxon por fold (B vs A): p={p_wilcoxon:.4f}")
except ValueError as e:
    print("Wilcoxon no aplicable:", e)

>On the per-fold Wilcoxon test (p = 0.0625). With 5 folds, the Wilcoxon signed-rank test has a minimum possible p-value of 0.0625 (= 1/2⁴): even if B beat A in all 5 folds, it could not fall below that threshold. It is therefore inconclusive due to lack of power. The evidence that B > A is provided by the paired bootstrap (ΔPR-AUC ≈ +0.052, 95 % CI not crossing 0, p ≈ 0), which does not depend on the number of folds.


In [ ]:
# Bar chart with the confidence intervals
tabla_ordenada = tabla.sort_values("PR_AUC")
error_izq = tabla_ordenada["PR_AUC"] - tabla_ordenada["IC95_lo"]
error_der = tabla_ordenada["IC95_hi"] - tabla_ordenada["PR_AUC"]

fig, ax = plt.subplots(figsize=(9, 4))
ax.barh(tabla_ordenada["modelo"], tabla_ordenada["PR_AUC"],
        xerr=[error_izq, error_der], capsize=4,
        color=["#bbbbbb", "#88aabb", "#e8a33d", "#d1495b"], edgecolor="white")
ax.axvline(df["target"].mean(), color="gray", linestyle="--", linewidth=1,
           label=f"azar = {df['target'].mean():.3f}")
ax.set_xlabel("PR-AUC (con IC 95%)")
ax.set_title("Comparación de modelos de propensión")
ax.legend()
plt.tight_layout()
plt.show()

## 12 · Which variables matter most? (Variant B)


In [ ]:
# Variable importance (model B on ALL the data, for inspection only — NOT an OOF metric).
# Here the embeddings PCA is fitted globally: this is legitimate because no validation metric is
# reported, only which variables the model uses.
pca_imp = PCA(n_components=N_EMB, random_state=42).fit(EMB_RAW_MAT)
emb_imp = pca_imp.transform(EMB_RAW_MAT)[ROW_PROD]
emb_names = [f"emb_pca_{i}" for i in range(emb_imp.shape[1])]

X_imp = np.hstack([df[feats_nivel2B].values.astype(float), emb_imp])
nombres_imp = feats_nivel2B + emb_names
cat_idx_B = [nombres_imp.index(c) for c in CAT_COLS]

modelo_final = lgb.LGBMClassifier(**LGBM_PARAMS)
modelo_final.fit(X_imp, y, categorical_feature=cat_idx_B)

importancia = pd.Series(modelo_final.feature_importances_, index=nombres_imp).sort_values()

fig, ax = plt.subplots(figsize=(8, 7))
importancia.plot(kind="barh", ax=ax, color="#d1495b")
ax.set_title("Importancia de cada variable (Variante B)")
ax.set_xlabel("Nº de veces que el modelo la usa")
plt.tight_layout()
plt.show()

print("Top 10 variables más importantes:")
print(importancia.sort_values(ascending=False).head(10))

## 13 · Saving the propensity scores

We export `propensity_scores.csv`. The final score is that of Variant B. `p_real` applies the click-rate
correction (so that the probabilities are realistic; it does not change the ordering).


In [ ]:
salida = df[["id_event", "id_user", "id_product", "sector", "target", "product_new"]].copy()
salida["p_sector"] = np.round(p_sector, 6)
salida["p_cond"] = np.round(p_cond_B, 6)
salida["p_model"] = np.round(p_cond_B, 6)   # adopted FLAT model (better PR-AUC than the hierarchical one)
salida["p_real"] = np.round(correct_prior(p_cond_B, rho_real, rho_train), 6)
salida["model_used"] = "flat_user_product_emb"

salida.to_csv(PROCESSED_PATH / "propensity_scores.csv", index=False)
print("Guardado: propensity_scores.csv", salida.shape)
salida.head()

## Summary of decisions (M2)

| # | Decision | Why |
|---|----------|-----|
| Hierarchical | sector → product, score = `p_sector × p_producto` | Mimics the two-step purchase decision and allows products to be ranked |
| GroupKFold by user | validate without a user being in both training and validation | Avoids the inflated score (data leakage) |
| PR-AUC + bootstrap + Wilcoxon | compare with intervals and a test, not by eye | A difference of means does not prove that one model is better |
| Binned age | age groups (production bands) | Interpretable and required for the inverse model (notebook 10) |
| Variant B > A | add product variables + subject-line embeddings | Learns user×product; beats production and popularity |

**Output:** `data/processed/propensity_scores.csv`


## 14 · Probability calibration

AUC measures whether the model ranks well, but not whether its probabilities are realistic. A probability is calibrated if, when the model predicts 0.30, roughly 30 % of those cases are clicks. It is measured with the Brier score (lower = better) and the reliability curve (ideally on the diagonal).


In [ ]:
from sklearn.metrics import brier_score_loss
from sklearn.calibration import calibration_curve

p_model_arr = p_cond_B                                       # probability on the training scale
p_real_arr = correct_prior(p_cond_B, rho_real, rho_train)    # corrected to the real prior

print("Brier score p_model (escala train):", round(brier_score_loss(y, p_model_arr), 5))
print("Brier score p_real  (escala prod) :", round(brier_score_loss(y, p_real_arr), 5))

# Reliability curve on p_model (10 uniform bins; smaller deviation from the diagonal = better)
frac_reales, prob_media = calibration_curve(y, p_model_arr, n_bins=10, strategy="uniform")
fig, ax = plt.subplots(figsize=(5, 5))
ax.plot([0, 1], [0, 1], "--", color="gray", label="calibracion perfecta")
ax.plot(prob_media, frac_reales, "o-", color="#d1495b", label="modelo")
ax.set_xlabel("Probabilidad media predicha")
ax.set_ylabel("Fraccion real de clics")
ax.set_title("Curva de fiabilidad (M2)")
ax.legend()
plt.tight_layout(); plt.show()

## 14.bis · Sensitivity to the real prior (ρ_real)

The real prior (production click rate) is not known exactly; it is assumed to be ≈ 2 %. The effect of it being 1 % or 3 % is checked. The prior correction is monotonic (it shifts the probabilities without reordering the users), so the ranking power (ROC-AUC, PR-AUC) should not change and only the absolute level of the probabilities would move.


In [ ]:
# Sensitivity to the real prior: 1% / 2% / 3%. The correction is monotonic -> ROC/PR-AUC invariant.
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss

filas_prior = []
for rho in [0.01, 0.02, 0.03]:
    pr = correct_prior(p_cond_B, rho, rho_train)
    filas_prior.append({
        "rho_real": rho,
        "media_p_real": round(float(pr.mean()), 4),
        "ROC-AUC": round(roc_auc_score(y, pr), 4),
        "PR-AUC": round(average_precision_score(y, pr), 4),
        "Brier": round(brier_score_loss(y, pr), 5),
    })
tabla_prior = pd.DataFrame(filas_prior)
print("=== Sensibilidad al prior real (rho_real) ===")
print(tabla_prior.to_string(index=False))
print()
print("ROC-AUC y PR-AUC son IDENTICOS en los tres casos: la capacidad de ordenar no depende del prior.")
print("Solo cambia el nivel absoluto (media_p_real sigue al prior asumido).")

## 15 · *Warm* model (with behaviour) for users with history

The earlier M2 is *cold*: it uses only demographics, valid for anyone but with little signal. For users
**with history** we can add their **past behaviour** (number of events, clicks, click-rate, sectors,
recency), computed **on the training set only** (no leakage). So that the *cold* vs *warm* comparison does not depend on
a single arbitrary temporal cut-off, we use **temporal cross-validation** (`TimeSeriesSplit`, expanding
window): several successive cut-offs in which training is the past and test the immediate future. At each
cut-off behaviour is recomputed on its training set only and evaluated on the test events of users who
already had history. We report the **mean ± standard deviation** across folds.


In [ ]:
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import TimeSeriesSplit

# Bring in each event's timestamp and sort chronologically
eventos_ts = events[["id_event", "timestamp"]].copy()
eventos_ts["timestamp"] = pd.to_datetime(eventos_ts["timestamp"])
dft = df.merge(eventos_ts, on="id_event", how="left").sort_values("timestamp").reset_index(drop=True)

BEH = ['b_eventos', 'b_clicks', 'b_click_rate', 'b_sectores', 'b_recencia']

def comportamiento_train(train_w):
    """Computes per-user behaviour using ONLY the training set (without looking at the future)."""
    fin = train_w['timestamp'].max()
    comp = train_w.groupby('id_user').agg(
        b_eventos=('id_event', 'count'), b_clicks=('target', 'sum'),
        b_sectores=('sector', 'nunique'), ultima=('timestamp', 'max')).reset_index()
    comp['b_click_rate'] = comp['b_clicks'] / comp['b_eventos']
    comp['b_recencia'] = (fin - comp['ultima']).dt.days
    return comp[['id_user'] + BEH]

def entrenar_evaluar(train_w, te, feats):
    cat_idx = [feats.index(c) for c in ['demo_cluster'] if c in feats]
    m = lgb.LGBMClassifier(**LGBM_PARAMS)
    m.fit(train_w[feats].values.astype(float), train_w['target'].values, categorical_feature=cat_idx)
    return m.predict_proba(te[feats].values.astype(float))[:, 1]

def evaluar_fold(train_w, test_w):
    """Returns PR-AUC/ROC of cold and warm on the test users WITH history."""
    comp = comportamiento_train(train_w)
    train_w = train_w.merge(comp, on='id_user', how='left')
    test_w = test_w.merge(comp, on='id_user', how='left')
    for c in BEH:
        train_w[c] = train_w[c].fillna(0); test_w[c] = test_w[c].fillna(0)
    hist = set(comp[comp['b_eventos'] > 0]['id_user'])
    te = test_w[test_w['id_user'].isin(hist)].copy()
    if te['target'].sum() < 5 or len(te) < 20:
        return None
    yw = te['target'].values
    pc = entrenar_evaluar(train_w, te, USER_FEATS)
    pw = entrenar_evaluar(train_w, te, USER_FEATS + BEH)
    return {'n': len(te), 'cr': float(yw.mean()),
            'cold_pr': average_precision_score(yw, pc), 'cold_roc': roc_auc_score(yw, pc),
            'warm_pr': average_precision_score(yw, pw), 'warm_roc': roc_auc_score(yw, pw)}

# --- Temporal cross-validation (expanding window) ---
tscv = TimeSeriesSplit(n_splits=5)
res = []
for k, (idx_tr, idx_te) in enumerate(tscv.split(dft)):
    r = evaluar_fold(dft.iloc[idx_tr].copy(), dft.iloc[idx_te].copy())
    if r is None:
        print(f"fold {k}: descartado (pocos positivos)"); continue
    res.append(r)
    print(f"fold {k}: n_test_hist={r['n']:5d}  click_rate={r['cr']:.3f}  |  "
          f"cold PR={r['cold_pr']:.4f}  warm PR={r['warm_pr']:.4f}  d={r['warm_pr']-r['cold_pr']:+.4f}")

cp = np.array([x['cold_pr'] for x in res]); wp = np.array([x['warm_pr'] for x in res])
cr = np.array([x['cold_roc'] for x in res]); wr = np.array([x['warm_roc'] for x in res])
d = wp - cp
print()
print(f"COLD  PR-AUC = {cp.mean():.4f} +/- {cp.std():.4f}   ROC = {cr.mean():.4f} +/- {cr.std():.4f}")
print(f"WARM  PR-AUC = {wp.mean():.4f} +/- {wp.std():.4f}   ROC = {wr.mean():.4f} +/- {wr.std():.4f}")
print(f"Delta PR-AUC (warm - cold) = {d.mean():+.4f} +/- {d.std():.4f}  "
      f"(folds con mejora: {int((d>0).sum())}/{len(d)})")
print("-> El comportamiento mejora la propension de forma consistente en todos los cortes temporales.")

## 16 · Ablation study — contribution of the synthetic variables

The ablation study removes groups of variables and measures the change in performance while keeping everything else constant (same GroupKFold by user, same PR-AUC metric with bootstrap CI). A flat model is used (LightGBM directly on the click, via `predecir_oof`) to isolate the contribution of each group:

- Random synthetic: `labor_status`, `civil_status`, `tiene_coche`, `size_hogar`, `num_room` (sampled from area distributions, noise at the individual level).
- Real user: `age_cat` (real age) plus `ipa_class`, `mun_type`, `distance_type` (geographic, derived from the postcode).
- Product: sector, category, CPL, marketing attributes and subject-line embeddings.

Each configuration is compared with the complete one (Δ with 95% CI and p-value from paired bootstrap).


In [ ]:
# --- Ablation: contribution of each variable group (flat OOF model, GroupKFold by user) ---
SINT_ALEATORIAS = ["labor_status_enc", "civil_status_enc", "tiene_coche_enc", "size_hogar_enc", "num_room_enc"]
REAL_USER       = ["age_cat", "ipa_class", "mun_type", "distance_type"]
PROD_FEATS      = ["sector_code", "prod_cat_code"] + PROD_NUM_FEATS   # embeddings are added with use_emb=True

# (list of features, add subject-line embeddings?)
configs = {
    "Completo (usuario+producto)":  (USER_FEATS + PROD_FEATS, True),
    "Sin sinteticas aleatorias":    ([f for f in USER_FEATS if f not in SINT_ALEATORIAS] + PROD_FEATS, True),
    "Solo real usuario + producto": (REAL_USER + PROD_FEATS, True),
    "Solo producto":                (PROD_FEATS, True),
    "Solo usuario (todo)":          (USER_FEATS, False),
}

oof_ablation = {nombre: predecir_oof(feats, use_emb=ue) for nombre, (feats, ue) in configs.items()}

base = oof_ablation["Completo (usuario+producto)"]
filas = []
for nombre, prob in oof_ablation.items():
    media, lo, hi = bootstrap_pr_auc(prob)
    feats, ue = configs[nombre]
    if nombre == "Completo (usuario+producto)":
        delta = "-"
    else:
        d, dlo, dhi, pval = bootstrap_diferencia(prob, base)   # config - complete
        signif = "" if (dlo <= 0 <= dhi) else " *"
        delta = f"{d:+.4f} [{dlo:+.4f}, {dhi:+.4f}] p={pval:.3f}{signif}"
    filas.append({
        "Configuracion": nombre, "n_vars": len(feats) + (N_EMB if ue else 0),
        "PR-AUC": round(media, 4), "IC95%": f"[{lo:.3f}, {hi:.3f}]",
        "ROC-AUC": round(roc_auc_score(y, prob), 4),
        "Delta vs completo": delta,
    })
ablation_df = pd.DataFrame(filas)
display(ablation_df)
print()
print("* = diferencia significativa (IC95% no cruza 0). PR-AUC azar (prevalencia) =", round(y.mean(), 3))

## 16.bis · Baseline by per-category click rate

The ablation study indicates that most of the signal is in the product. The reference baseline assigns each pair the historical click rate of the product's category (frequency table). It is computed OOF (the rate is estimated on each fold's training set only, GroupKFold by user) and compared with the flat model. If the model beats it, its value goes beyond encoding the category.


In [ ]:
# OOF per-category CTR baseline (GroupKFold by user): the category click rate,
# estimated using ONLY each fold's training set. Reference baseline for 'product propensity'.
oof_cat = np.zeros(len(y))
oof_sec = np.zeros(len(y))
cv_b = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
for tr, va in cv_b.split(np.zeros(len(y)), y, groups):
    gmean = y[tr].mean()
    ctr_cat = df.iloc[tr].groupby('product_new')['target'].mean()
    ctr_sec = df.iloc[tr].groupby('sector')['target'].mean()
    oof_cat[va] = df.iloc[va]['product_new'].map(ctr_cat).fillna(gmean).values
    oof_sec[va] = df.iloc[va]['sector'].map(ctr_sec).fillna(gmean).values

print('Prevalencia (azar) =', round(float(y.mean()), 4))
print(f'Baseline CTR-por-SECTOR    : PR-AUC={pr_auc(oof_sec):.4f}  ROC={roc_auc_score(y, oof_sec):.4f}')
print(f'Baseline CTR-por-CATEGORIA : PR-AUC={pr_auc(oof_cat):.4f}  ROC={roc_auc_score(y, oof_cat):.4f}')
print(f'Modelo PLANO (LightGBM)    : PR-AUC={pr_auc(p_cond_B):.4f}  ROC={roc_auc_score(y, p_cond_B):.4f}')
d, lo, hi, pv = bootstrap_diferencia(p_cond_B, oof_cat)
print(f'delta(flat - CTR_categoria) = {d:+.4f}  IC95%[{lo:+.4f},{hi:+.4f}]  p={pv:.4f}')
print('-> El modelo bate a la tabla de frecuencias por categoria; la mejora viene del texto del asunto (embeddings).')

## 17 · Algorithm comparison: flat vs hierarchical (including a neural network)

Each algorithm (LightGBM, **MLP neural network**, HistGradientBoosting, Random Forest and logistic regression) is
evaluated in **two ways** on the same GroupKFold-by-user partitions: as a **flat model** (predicting the
click directly) and as a **hierarchical model** (sector level $\times$ product level). This shows, for each
model, whether the hierarchical structure helps or hurts, with a panel of metrics (PR-AUC, ROC-AUC, Brier, LogLoss,
F1, Balanced Accuracy and MCC).


In [ ]:
# ============================================================
# 17 · Algorithm comparison: FLAT vs HIERARCHICAL (including a NEURAL NETWORK)
# Each algorithm is evaluated in two ways, on the SAME GroupKFold-by-user partitions:
#   - FLAT:        a model predicts P(click | user, product) directly.
#   - HIERARCHICAL: sector level P(click|u,sector) x product level P(click|u,sector,prod).
# ============================================================
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import (average_precision_score, roc_auc_score, brier_score_loss,
                             log_loss, f1_score, balanced_accuracy_score, matthews_corrcoef)

feats_flat = feats_nivel2B                  # user + sector + product (+ embeddings)
feats_sec  = USER_FEATS + ["sector_code"]   # level 1: user + sector

def oof_generic(feats, use_emb, make_model, scale):
    """OOF GroupKFold by user for ANY model. Categoricals -> one-hot;
    if scale=True, numericals are imputed(median)+standardised within each fold (no leakage).
    Subject-line embeddings with per-fold PCA when use_emb=True."""
    num_cols = [f for f in feats if f not in CAT_COLS]
    cat_cols = [f for f in feats if f in CAT_COLS]
    Xn = df[num_cols].values.astype(float)
    Xc = pd.get_dummies(df[cat_cols].astype("category")).values.astype(float) if cat_cols else np.zeros((len(df),0))
    oof = np.zeros(len(y))
    cv = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
    for tr, va in cv.split(Xn, y, groups):
        if use_emb:
            prods_tr = np.unique(ROW_PROD[tr]); k = min(N_EMB, len(prods_tr)-1)
            pca = PCA(n_components=k, random_state=42).fit(EMB_RAW_MAT[prods_tr])
            emb = pca.transform(EMB_RAW_MAT)[ROW_PROD]; etr, eva = emb[tr], emb[va]
        else:
            etr = np.zeros((len(tr),0)); eva = np.zeros((len(va),0))
        ntr, nva = Xn[tr], Xn[va]
        if scale:
            imp = SimpleImputer(strategy="median").fit(ntr)
            ss = StandardScaler().fit(imp.transform(ntr))
            ntr = ss.transform(imp.transform(ntr)); nva = ss.transform(imp.transform(nva))
            if use_emb:
                se = StandardScaler().fit(etr); etr = se.transform(etr); eva = se.transform(eva)
        Xtr = np.hstack([ntr, Xc[tr], etr]); Xva = np.hstack([nva, Xc[va], eva])
        m = make_model(); m.fit(Xtr, y[tr]); oof[va] = m.predict_proba(Xva)[:, 1]
    return oof

ALGOS = {
    "LightGBM":             (lambda: lgb.LGBMClassifier(**LGBM_PARAMS), False),
    "Red neuronal (MLP)":   (lambda: MLPClassifier(hidden_layer_sizes=(64,32), alpha=1e-3, batch_size=256,
                                max_iter=300, early_stopping=True, n_iter_no_change=10, random_state=42), True),
    "HistGradientBoosting": (lambda: HistGradientBoostingClassifier(learning_rate=0.05, max_iter=300, random_state=42), False),
    "Random Forest":        (lambda: RandomForestClassifier(n_estimators=300, min_samples_leaf=20, n_jobs=-1, random_state=42), True),
    "Regresion logistica":  (lambda: LogisticRegression(max_iter=1000), True),
}

thr = float(y.mean())
def metricas(oof):
    pred = (oof >= thr).astype(int)
    return {"PR-AUC": round(average_precision_score(y, oof), 4),
            "ROC-AUC": round(roc_auc_score(y, oof), 4),
            "Brier": round(brier_score_loss(y, oof), 4),
            "LogLoss": round(log_loss(y, oof), 4),
            "F1": round(f1_score(y, pred), 4),
            "BalAcc": round(balanced_accuracy_score(y, pred), 4),
            "MCC": round(matthews_corrcoef(y, pred), 4)}

res_plano, res_hier, deltas = [], [], []
store = {"id_event": df["id_event"].values, "target": y}
for nombre, (mk, sc) in ALGOS.items():
    p_cond = oof_generic(feats_flat, True,  mk, sc)   # FLAT (product level)
    p_sec  = oof_generic(feats_sec,  False, mk, sc)   # sector level
    hier   = p_sec * p_cond                            # HIERARCHICAL
    store[nombre + "|plano"] = np.round(p_cond, 6)
    store[nombre + "|jerarquico"] = np.round(hier, 6)
    mp, mh = metricas(p_cond), metricas(hier)
    res_plano.append({"Algoritmo": nombre, **mp})
    res_hier.append({"Algoritmo": nombre, **mh})
    deltas.append({"Algoritmo": nombre, "PR-AUC plano": mp["PR-AUC"],
                   "PR-AUC jerarquico": mh["PR-AUC"], "Delta (jer-plano)": round(mh["PR-AUC"]-mp["PR-AUC"], 4)})

pd.DataFrame(store).to_csv(PROCESSED_PATH / "m2_algo_oof.csv", index=False)
df_plano = pd.DataFrame(res_plano).sort_values("PR-AUC", ascending=False)
df_hier  = pd.DataFrame(res_hier).sort_values("PR-AUC", ascending=False)
df_delta = pd.DataFrame(deltas).sort_values("PR-AUC plano", ascending=False)

print("########## MODELO PLANO (PR-AUC, GroupKFold por usuario; umbral F1/BalAcc/MCC = %.3f) ##########" % thr)
print(df_plano.to_string(index=False))
print("\n########## MODELO JERARQUICO (p_sector x p_producto) ##########")
print(df_hier.to_string(index=False))
print("\n########## EFECTO DE LA JERARQUIA (Delta PR-AUC = jerarquico - plano) ##########")
print(df_delta.to_string(index=False))
print("\nBrier y LogLoss: menor = mejor. El resto: mayor = mejor.")


## 18 · Per-algorithm hyperparameter tuning (fair comparison) — PENDING EXECUTION

So that the comparison between algorithms is **fair** (and does not penalise the neural network for using
default configurations), the hyperparameters of each model are tuned via a bounded grid search, over the
**same GroupKFold partitions** and the same flat model. **Warning:** it is costly
(~30–60 min: each configuration retrains 5 folds). After running it, update the comparison table in
memory with the tuned numbers and the sentence "after tuning the hyperparameters, the result holds".


In [ ]:
# ============================================================
# 18 · Per-algorithm hyperparameter tuning (FAIR comparison)   [PENDING EXECUTION]
# Reuses oof_generic / metricas / feats_flat from section 17. COSTLY (~30-60 min).
# ============================================================
from itertools import product

def mejor_config(maker, grid, scale):
    """Evaluates each grid combination with oof_generic (PR-AUC) and returns the best one."""
    best = (-1.0, None, None)
    nombres = list(grid.keys())
    for combo in product(*grid.values()):
        params = dict(zip(nombres, combo))
        oof = oof_generic(feats_flat, True, lambda: maker(params), scale)
        ap = average_precision_score(y, oof)
        if ap > best[0]:
            best = (ap, params, oof)
    return best

REJILLAS = {
    "LightGBM": (
        lambda p: lgb.LGBMClassifier(objective="binary", random_state=42, verbose=-1, n_jobs=-1,
                                     subsample=0.8, colsample_bytree=0.8, min_child_samples=20, **p),
        {"num_leaves": [15, 31, 63], "learning_rate": [0.03, 0.05, 0.1], "n_estimators": [200, 400]}, False),
    "Red neuronal (MLP)": (
        lambda p: MLPClassifier(max_iter=300, early_stopping=True, n_iter_no_change=10,
                                batch_size=256, random_state=42, **p),
        {"hidden_layer_sizes": [(64, 32), (128, 64), (64,)], "alpha": [1e-4, 1e-3, 1e-2]}, True),
    "HistGradientBoosting": (
        lambda p: HistGradientBoostingClassifier(random_state=42, **p),
        {"learning_rate": [0.03, 0.05, 0.1], "max_iter": [200, 400], "max_leaf_nodes": [15, 31]}, False),
    "Random Forest": (
        lambda p: RandomForestClassifier(n_jobs=-1, random_state=42, **p),
        {"n_estimators": [300, 600], "max_depth": [None, 10, 20], "min_samples_leaf": [5, 20]}, True),
    "Regresion logistica": (
        lambda p: LogisticRegression(max_iter=1000, **p),
        {"C": [0.1, 1.0, 10.0]}, True),
}

filas_tuned = []
for nombre, (maker, grid, sc) in REJILLAS.items():
    ap, params, oof = mejor_config(maker, grid, sc)
    filas_tuned.append({"Algoritmo": nombre, **metricas(oof), "mejores_params": params})
tabla_tuned = pd.DataFrame(filas_tuned).sort_values("PR-AUC", ascending=False)
print("=== Comparacion TUNEADA (mejor config por algoritmo, modelo plano) ===")
print(tabla_tuned.to_string(index=False))
